## Multi-Accent and Multi-Lingual Voice Clone Demo with MeloTTS

In [1]:
import os
import torch
from openvoice import se_extractor
from openvoice.api import ToneColorConverter

d:\GITHUB\stable-diffusion-webui\venv_gpu\lib\site-packages\pydub\utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


Importing the dtw module. When using in academic works please cite:
  T. Giorgino. Computing and Visualizing Dynamic Time Warping Alignments in R: The dtw Package.
  J. Stat. Soft., doi:10.18637/jss.v031.i07.



### Initialization

In this example, we will use the checkpoints from OpenVoiceV2. OpenVoiceV2 is trained with more aggressive augmentations and thus demonstrate better robustness in some cases.

In [2]:
ckpt_converter = 'checkpoints_v2/converter'
device = "cuda:0" if torch.cuda.is_available() else "cpu"
output_dir = 'outputs_v2'

tone_color_converter = ToneColorConverter(f'{ckpt_converter}/config.json', device=device)
tone_color_converter.load_ckpt(f'{ckpt_converter}/checkpoint.pth')

os.makedirs(output_dir, exist_ok=True)

d:\GITHUB\stable-diffusion-webui\venv_gpu\lib\site-packages\torch\nn\utils\weight_norm.py:30: UserWarning: torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.
  warnings.warn("torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.")


Loaded checkpoint 'checkpoints_v2/converter/checkpoint.pth'
missing/unexpected keys: [] []


### Obtain Tone Color Embedding
We only extract the tone color embedding for the target speaker. The source tone color embeddings can be directly loaded from `checkpoints_v2/ses` folder.

In [8]:

reference_speaker = 'D:/GITHUB/MP-SENet/tvboy1_denoised.wav' # This is the voice you want to clone
target_se, audio_name = se_extractor.get_se(reference_speaker, tone_color_converter, vad=True)

#my_speaker = 'inputs/train/medium.wav'
#my_se, my_audioname = se_extractor.get_se(my_speaker, tone_color_converter, vad=True)

OpenVoice version: v2


FileNotFoundError: [WinError 2] 系统找不到指定的文件。

In [6]:
torch.save(target_se, "inputs/voice/tvboy2_denoised.pth")
# torch.save(my_se, "inputs/voice/mymedium_vad.pth")

In [6]:
my_se = torch.load(f'inputs/voice/mylong.pth', map_location=device)

#### Use MeloTTS as Base Speakers

MeloTTS is a high-quality multi-lingual text-to-speech library by @MyShell.ai, supporting languages including English (American, British, Indian, Australian, Default), Spanish, French, Chinese, Japanese, Korean. In the following example, we will use the models in MeloTTS as the base speakers. 

In [9]:
tone_color_converter.convert(
            audio_src_path=f'inputs/test/test3.wav', 
            src_se=my_se, 
            tgt_se=target_se, 
            output_path=f'inputs/output/long2tvboy2_test1.wav')

In [9]:
from melo.api import TTS

texts = {
    #'EN_NEWEST': "Did you ever hear a folk tale about a giant turtle?",  # The newest English base speaker model
    #'EN': "Did you ever hear a folk tale about a giant turtle?",
    #'ES': "El resplandor del sol acaricia las olas, pintando el cielo con una paleta deslumbrante.",
    #'FR': "La lueur dorée du soleil caresse les vagues, peignant le ciel d'une palette éblouissante.",
    'ZH': "在我妈生我的前一天我就被妈妈抛弃了，我当时哭了三天三夜。为了防止此类事件不再发生，我一气之下把我婆婆的儿媳妇赶出了家门。时隔多年，功夫不负有心人，我终于找到了我失散多年的老婆，从此走出了正常的家庭。我虽然命运多舛，但足够好运。我没有生育能力老婆却给我生了个孩子。很欣慰，儿子高考考了785分，惊呆所有人。我坚信，万事皆有奇迹，只要不断奋斗就有吃不完的苦，为此，我要红烧排骨、烤全羊、麻婆豆腐、烤乳猪来丰富我家狗的饮食。学到老，玩到老，每天坚持用量子波动法放生癌细胞。我懂得养生，每天喝3罐中药预防视力低于6.0。我一生积水成德，没想到年过七旬却遭遇背叛，有天晚上我继父偷偷地溜进我的房间，给我深深的上了一课，导致70岁的我患上了20岁才得的病。我忍无可忍，做出了违背祖宗的决定，在他70岁大寿的时候我撅翻桌子，他肯定不懂得自己怀胎10月生下来的孩子竟然不是自己的痛苦。欲知后续，给我免费的赞，我将诉说我的复仇计划。",
    #'JP': "彼は毎朝ジョギングをして体を健康に保っています。",
    #'KR': "안녕하세요! 오늘은 날씨가 정말 좋네요.",
}


src_path = f'{output_dir}/tmp.wav'

# Speed is adjustable
speed = 1.0

for language, text in texts.items():
    model = TTS(language=language, device=device)
    speaker_ids = model.hps.data.spk2id
    
    for speaker_key in speaker_ids.keys():
        speaker_id = speaker_ids[speaker_key]
        speaker_key = speaker_key.lower().replace('_', '-')
        
        source_se = torch.load(f'checkpoints_v2/base_speakers/ses/{speaker_key}.pth', map_location=device)
        model.tts_to_file(text, speaker_id, src_path, speed=speed)
        save_path = f'{output_dir}/output_v2_{speaker_key}.wav'

        # Run the tone color converter
        encode_message = "@MyShell"
        tone_color_converter.convert(
            audio_src_path=src_path, 
            src_se=source_se, 
            tgt_se=target_se, 
            output_path=save_path,
            message=encode_message)

 > Text split to sentences.
在我妈生我的前一天我就被妈妈抛弃了,
我当时哭了三天三夜. 为了防止此类事件不再发生,
我一气之下把我婆婆的儿媳妇赶出了家门.
时隔多年, 功夫不负有心人,
我终于找到了我失散多年的老婆,
从此走出了正常的家庭.
我虽然命运多舛, 但足够好运.
我没有生育能力老婆却给我生了个孩子.
很欣慰, 儿子高考考了785分,
惊呆所有人. 我坚信, 万事皆有奇迹,
只要不断奋斗就有吃不完的苦,
为此, 我要红烧排骨、烤全羊、麻婆豆腐、烤乳猪来丰富我家狗的饮食.
学到老, 玩到老, 每天坚持用量子波动法放生癌细胞.
我懂得养生, 每天喝3罐中药预防视力低于6.
0. 我一生积水成德, 没想到年过七旬却遭遇背叛,
有天晚上我继父偷偷地溜进我的房间,
给我深深的上了一课, 导致70岁的我患上了20岁才得的病.
我忍无可忍, 做出了违背祖宗的决定,
在他70岁大寿的时候我撅翻桌子,
他肯定不懂得自己怀胎10月生下来的孩子竟然不是自己的痛苦.
欲知后续, 给我免费的赞,
我将诉说我的复仇计划.
 > ===========================


100%|██████████| 22/22 [00:04<00:00,  4.60it/s]
